# 📂❓🔄✅ <span style="color: white; background-color: Indigo"><b> Validação de Centro de Custo</b></span></p>

🧩 <span style="color: Orchid"><b> 1- Carregamento e Tratamento das Bases </b></span></p>
- A automação utiliza duas bases principais:
    - HC — base de colaboradores com cargo, empresa, registro, datas e centros de custo  
    - CENTRO DE CUSTO.xlsx — tabela de referência com nome e código dos centros
- O processo inicia com:
    - Importação do HC  
    - Importação do arquivo de referência  
    - Tratamento das colunas  
    - Conversão da data de admissão  
    - Junção (left join) por cod_centro_custo
- Tudo isso garante uma base consolidada, completa e consistente

🔍 <span style="color: Orchid"><b> 2- Filtragem de Colaboradores Relevantes </b></span></p>
- A automação aplica regras fundamentais apenas para colaboradores ativos
- Exclui todos com data_demissao preenchida
- Remove cargos que contenham “ADMINISTRATIVO”
- Exclusão da empresa interna (999)
- Evita registros que não representem operação real
- O resultado é uma base limpa para análise

🧮 <span style="color: Orchid"><b> 3- Determinação do Centro de Custo “Esperado” por Cargo </b></span></p>
- Para cada cargo, a automação identifica o centro de custo mais frequente (modo)
- Esse centro representa o padrão histórico — ou seja, o centro “esperado”
-  script então compara:
    - Centro de Custo Atual  
    - Centro de Custo Esperado
- E identifica divergências automaticamente

⚠️ <span style="color: Orchid"><b> 4- Identificação de Divergências </b></span></p>
- Cada colaborador recebe um indicador de divergência
- Somente colaboradores com diferença entre atual × esperado são enviados ao relatório
- Datas são convertidas para DD/MM/AAAA  
- Cargos e unidades são organizados  
- A base final é ordenada por cargo e centro de custo atual

🌐 <span style="color: Orchid"><b> 5- Geração do Relatório Final em HTML </b></span></p>
- O relatório gerado possui:
    - Título: "Validação de Centros de Custo – Colaboradores com Divergências"
    - Resumo inicial: “Total de divergências encontradas: X registros.”
- Tabela estruturada com as colunas:
    - Registro  
    - Nome  
    - Data de Admissão  
    - Empresa  
    - Cargo  
    - Cód. Centro de Custo  
    - Centro de Custo Atual  
    - Centro de Custo Esperado

# Importar Bibliotecas

In [1]:
import pandas as pd
from datetime import datetime, date
import os
from openpyxl import load_workbook, Workbook

# Caminhos dos Arquivos

In [2]:
caminho_centro_custo = r"X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\CENTRO DE CUSTO.xlsx"
caminho_hc = r"X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.4 - HC e Atestados Médicos\Controle_HC e Atestados.xlsx"
caminho_saida = r"X:\Gestão de Pessoas\Analytics\11 - Validações\11.3 - Centros de Custo"
path_registros_processos = r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'

# Carregamento da Base de Controle de Processos

In [3]:
registros_processos = pd.read_excel(path_registros_processos, sheet_name = "REGISTROS", engine='openpyxl')
wb_p = load_workbook(path_registros_processos)
ws_p = wb_p['REGISTROS']

# Controle de atualização de processo
id = 32
tempo_0 = [id, datetime.today(), 0]
ws_p.append(tempo_0)
wb_p.save(path_registros_processos)

# Tratamento da Base e Geração de HTML

In [4]:
try:
    # Ler arquivos Excel
    print("Carregando bases de dados...")
    df_centro_custo = pd.read_excel(caminho_centro_custo)
    df_hc = pd.read_excel(caminho_hc, sheet_name="HC")
    
    # Left join da base HC com CENTRO DE CUSTO usando cod_centro_custo
    print("Relacionando as bases...")
    df_merged = pd.merge(df_hc, df_centro_custo, on="cod_centro_custo", how="left", suffixes=('_hc', '_cc'))
    
    # Filtrar apenas ativos (data_rescisao nula ou vazia) E cod_empresa diferente de 999
    print("Filtrando colaboradores ativos...")
    df_ativos = df_merged[
        (df_merged['status'] == 'ATIVO') & 
        (df_merged['cod_empresa'] != 999) & 
        (df_merged['situacao'] != 'AFASTADO C/PROCESSAM')
    ]

    # Remover cargos com 'ADMINISTRATIVO' na coluna cargo_completo
    df_ativos = df_ativos[~df_ativos['cargo_completo'].str.upper().str.contains('ADMINISTRATIVO|SECRETARIO', na=False)]

    # Converter data_admissao de formato numérico para DD/MM/AAAA
    print("Convertendo formato de data...")
    df_ativos['data_admissao'] = df_ativos['data_admissao'].dt.strftime('%d/%m/%Y')
    
    print(df_ativos['data_admissao'].dtype)  # Deve exibir: object
    print(df_ativos['data_admissao'].head())  # Deve exibir datas como strings

    print("Criando tabela de referência de cargos...")
    
    # Extrair o centro de custo mais frequente para cada cargo
    cargo_centro_correto = df_ativos.groupby('cargo_completo')['centro_custo_cc'].agg(
        lambda x: x.value_counts().index[0] if len(x.value_counts()) > 0 else None
    ).to_dict()
    
    # Identificar divergências: comparar centro de custo atual com o correto
    print("Identificando divergências...")
    df_ativos['centro_custo_correto'] = df_ativos['cargo_completo'].map(cargo_centro_correto)
    df_ativos['tem_divergencia'] = df_ativos['centro_custo_cc'] != df_ativos['centro_custo_correto']
    
    # Filtrar apenas colaboradores com divergências
    df_relatorio = df_ativos[df_ativos['tem_divergencia']].copy()
    
    # Selecionar colunas para o relatório
    df_relatorio = df_relatorio[['registro', 'nome', 'data_admissao', 'empresa_resumo', 'cargo_completo', 'cod_centro_custo', 'centro_custo_cc', 'centro_custo_correto']]
    
    # Renomear colunas para melhor visualização
    df_relatorio = df_relatorio.rename(columns={
        'registro': 'Registro',
        'nome': 'Nome',
        'data_admissao': 'Data de Admissão',
        'empresa_resumo': 'Empresa',
        'cargo_completo': 'Cargo',
        'cod_centro_custo': 'Cód. Centro de Custo',
        'centro_custo_cc': 'Centro de Custo Atual',
        'centro_custo_correto': 'Centro Custo Esperado'
    })
    
    # Ordenar para facilitar a visualização
    df_relatorio = df_relatorio.sort_values(by=['Cargo', 'Centro de Custo Atual'])
    
    # Gerar data para o nome do arquivo
    data_hoje = datetime.now().strftime("%d-%m-%Y")
    nome_arquivo = f"VALIDAÇÃO DE CENTRO DE CUSTO_{data_hoje}.html"
    
    # Garantir que o diretório de saída existe
    os.makedirs(caminho_saida, exist_ok=True)
    caminho_completo = os.path.join(caminho_saida, nome_arquivo)
    
    # Gerar HTML
    print("Gerando relatório HTML...")
    html_header = f"""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <title>Validação de Centros de Custo</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 20px; color: #333; }}
            h1 {{ color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px; }}
            .info {{ margin-bottom: 20px; font-size: 14px; color: #555; }}
            table {{ border-collapse: collapse; width: 100%; font-size: 14px; }}
            th, td {{ border: 1px solid #ddd; padding: 10px; text-align: left; }}
            th {{ background-color: #3498db; color: white; position: sticky; top: 0; }}
            tr:nth-child(even) {{ background-color: #f9f9f9; }}
            tr:hover {{ background-color: #f1f1f1; }}
            .divergencia {{ background-color: #ffcccc; }}
        </style>
    </head>
    <body>
        <h1>Validação de Centros de Custo</h1>
        <div class="info">
            <p><strong>Data de execução:</strong> {datetime.now().strftime('%d/%m/%Y às %H:%M:%S')}</p>
            <p><strong>Total de divergências encontradas:</strong> {len(df_relatorio)} registros.</p>
        </div>
    """
    
    # Converter DataFrame para HTML sem o índice
    html_table = df_relatorio.to_html(index=False, escape=False, na_rep='-')
    html_footer = "</body></html>"
    
    html_completo = html_header + html_table + html_footer
    
    # Arquivo HTML
    with open(caminho_completo, 'w', encoding='utf-8') as f:
        f.write(html_completo)
    
    print(f"✅ Sucesso! Relatório gerado em: {caminho_completo}")
    print(f"Total de divergências: {len(df_relatorio)}")

except FileNotFoundError as e:
    print(f"❌ Erro: Arquivo ou diretório não encontrado. Verifique os caminhos.\nDetalhes: {str(e)}")
except KeyError as e:
    print(f"❌ Erro: Coluna não encontrada nas bases de dados.\nDetalhes: {str(e)}")
except Exception as e:
    print(f"❌ Erro inesperado durante a execução:\nDetalhes: {str(e)}")

Carregando bases de dados...
Relacionando as bases...
Filtrando colaboradores ativos...
Convertendo formato de data...
object
51     01/04/1983
89     08/01/1986
100    20/10/1986
129    01/06/1988
136    06/09/1988
Name: data_admissao, dtype: object
Criando tabela de referência de cargos...
Identificando divergências...
Gerando relatório HTML...
✅ Sucesso! Relatório gerado em: X:\Gestão de Pessoas\Analytics\11 - Validações\11.3 - Centros de Custo\VALIDAÇÃO DE CENTRO DE CUSTO_24-06-2026.html
Total de divergências: 4


# Resumo de Finalização do Processo

In [5]:
tempo_1 = [id, datetime.today(), 1]

print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

     ✅  Processo finalizado

     ⏱️   Tempo de execução:

   0:00:09.177499

----------------------------------------------------------------------------------------------------
